<a href="https://colab.research.google.com/github/fatmaemad8/RSRP_RSSI-Prediction/blob/main/RSRP_RSSI_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd

df = pd.read_csv("/ml.csv")
print(df.columns.tolist())

['timestamp', 'rel_time', 'longitude', 'latitude', 'altitude', 'pci', 'ci', 'is_connected', 'tac', 'earfcn', 'rsrp', 'rsrq', 'rssi', 'asu', 'ta']


In [10]:
import numpy as np

# تحويل كلمة Unavailable إلى قيمة فارغة حقيقية (NaN)
df = df.replace("Unavailable", np.nan)

print("عدد الصفوف والأعمدة:", df.shape)
print("\nأول 5 صفوف:")
print(df.head())
print("\nعدد القيم الفارغة في كل عمود:")
print(df.isna().sum())

عدد الصفوف والأعمدة: (56602, 15)

أول 5 صفوف:
      timestamp  rel_time  longitude   latitude  altitude  pci         ci  \
0  1.693070e+12  1.014608 -78.666622  35.786425     4.216  171  195616280   
1  1.693070e+12  1.014654 -78.666622  35.786425     4.216  192        NaN   
2  1.693070e+12  1.014688 -78.666622  35.786425     4.216   31        NaN   
3  1.693070e+12  1.014718 -78.666622  35.786425     4.216  198        NaN   
4  1.693070e+12  2.035097 -78.666622  35.786425     4.216  171  195616280   

   is_connected   tac  earfcn  rsrp  rsrq  rssi  asu   ta  
0             1  9995   66486  -104   -15   -71   36  NaN  
1             0   NaN   66486  -104   -11   -83   36  NaN  
2             0   NaN   66486  -113   -20   -83   27  NaN  
3             0   NaN   66486  -108   -15   -83   32  NaN  
4             1  9995   66486  -103   -15   -69   37  NaN  

عدد القيم الفارغة في كل عمود:
timestamp           0
rel_time            0
longitude           0
latitude            0
altitude    

In [11]:
missing_pct = (df.isna().sum() / len(df)) * 100
summary = pd.DataFrame({
    "نوع البيانات": df.dtypes,
    "عدد الفراغات": df.isna().sum(),
    "نسبة الفراغات %": missing_pct.round(2),
})
print(summary)

             نوع البيانات  عدد الفراغات  نسبة الفراغات %
timestamp         float64             0             0.00
rel_time          float64             0             0.00
longitude         float64             0             0.00
latitude          float64             0             0.00
altitude          float64             0             0.00
pci                 int64             0             0.00
ci                 object         50222            88.73
is_connected        int64             0             0.00
tac                object         50222            88.73
earfcn              int64             0             0.00
rsrp                int64             0             0.00
rsrq                int64             0             0.00
rssi                int64             0             0.00
asu                 int64             0             0.00
ta                 object          9076            16.03


In [12]:
df_clean = df.drop(columns=["ci", "tac"]).copy()
df_clean["ta"] = pd.to_numeric(df_clean["ta"], errors="coerce")

print(df_clean.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56602 entries, 0 to 56601
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     56602 non-null  float64
 1   rel_time      56602 non-null  float64
 2   longitude     56602 non-null  float64
 3   latitude      56602 non-null  float64
 4   altitude      56602 non-null  float64
 5   pci           56602 non-null  int64  
 6   is_connected  56602 non-null  int64  
 7   earfcn        56602 non-null  int64  
 8   rsrp          56602 non-null  int64  
 9   rsrq          56602 non-null  int64  
 10  rssi          56602 non-null  int64  
 11  asu           56602 non-null  int64  
 12  ta            47526 non-null  float64
dtypes: float64(6), int64(7)
memory usage: 5.6 MB
None


In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# 1. تنظيف البيانات
df_clean = df.drop(columns=["ci", "tac"]).copy()
df_clean["ta"] = pd.to_numeric(df_clean["ta"], errors="coerce")
df_clean["ta"] = df_clean["ta"].fillna(df_clean["ta"].median())

# 2. المعطيات (X) والهدفين (y_rsrp و y_rssi)
X = df_clean.drop(columns=["rsrp", "rssi", "asu"])
y_rsrp = df_clean["rsrp"]
y_rssi = df_clean["rssi"]

# 3. تقسيم البيانات مرة واحدة مع الحفاظ على نفس الصفوف للاثنين
X_train, X_test, y_rsrp_train, y_rsrp_test, y_rssi_train, y_rssi_test = (
    train_test_split(X, y_rsrp, y_rssi, test_size=0.2, random_state=42)
)

# 4. نموذج توقع RSRP
model_rsrp = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model_rsrp.fit(X_train, y_rsrp_train)
pred_rsrp = model_rsrp.predict(X_test)

print("=== نتائج توقع RSRP ===")
print("متوسط الخطأ (MAE):", round(mean_absolute_error(y_rsrp_test, pred_rsrp), 2))
print("معامل الجودة (R2):", round(r2_score(y_rsrp_test, pred_rsrp), 4))

# 5. نموذج توقع RSSI
model_rssi = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model_rssi.fit(X_train, y_rssi_train)
pred_rssi = model_rssi.predict(X_test)

print("\n=== نتائج توقع RSSI ===")
print("متوسط الخطأ (MAE):", round(mean_absolute_error(y_rssi_test, pred_rssi), 2))
print("معامل الجودة (R2):", round(r2_score(y_rssi_test, pred_rssi), 4))

=== نتائج توقع RSRP ===
متوسط الخطأ (MAE): 0.67
معامل الجودة (R2): 0.9914

=== نتائج توقع RSSI ===
متوسط الخطأ (MAE): 0.51
معامل الجودة (R2): 0.9906
